# Week 4 — Solutions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("../../assets/mplstyle/course.mplstyle")
sweep = pd.read_csv("../data/sweep_results.csv")


## Solution 1 — Slice plot

In [ ]:
params = ["lr", "weight_decay", "batch_size", "dropout", "optimizer"]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.3))
for ax, p in zip(axes, params):
    if p in ("lr", "weight_decay"):
        ax.set_xscale("log")
    if sweep[p].dtype == object:
        order = sweep.groupby(p)["val_loss"].mean().sort_values().index
        ax.scatter(sweep[p], sweep["val_loss"], s=36, alpha=0.7, c="#0072B2")
        ax.set_xticks(range(len(order)), order, rotation=20)
    else:
        ax.scatter(sweep[p], sweep["val_loss"], s=36, alpha=0.7,
                   c=sweep["val_loss"], cmap="viridis_r")
    ax.set(xlabel=p, ylabel="val_loss" if ax is axes[0] else "", title=p)
plt.tight_layout(); plt.show()


**Reading.** `lr` shows a clear U-shape: very small and very large learning
rates both push val_loss up, with a sweet spot near $10^{-3}$. `dropout` shows a softer
U-shape centred near 0.25. `optimizer` shows a clear ranking (`adamw < adam < sgd`).
`weight_decay` and `batch_size` are mostly flat — they don't drive the metric in this
search space.

## Solution 2 — Parallel coordinates

Reuse of the `parallel_coordinates` helper from the lab:

In [ ]:
def parallel_coordinates(df, axes_cols, metric_col, log_axes=(),
                         figsize=(11, 4.5), cmap="viridis"):
    fig, ax = plt.subplots(figsize=figsize)
    norms = {}
    for col in axes_cols:
        v = df[col]
        if v.dtype == object:
            cats = sorted(v.unique())
            mapping = {c: i / max(len(cats) - 1, 1) for i, c in enumerate(cats)}
            norms[col] = (v.map(mapping).values, cats, "cat")
        else:
            vals = np.log10(v.values) if col in log_axes else v.values
            lo, hi = vals.min(), vals.max()
            span = hi - lo if hi > lo else 1.0
            norms[col] = ((vals - lo) / span, (lo, hi), "log" if col in log_axes else "lin")
    metric = df[metric_col].values
    cmap = plt.get_cmap(cmap)
    m_lo, m_hi = metric.min(), metric.max()
    colors = cmap((metric - m_lo) / (m_hi - m_lo + 1e-12))
    xs = np.arange(len(axes_cols))
    for i in range(len(df)):
        ys = [norms[c][0][i] for c in axes_cols]
        ax.plot(xs, ys, color=colors[i], alpha=0.7, lw=1.5)
    ax.set_xticks(xs, axes_cols)
    ax.set_ylim(-0.02, 1.02); ax.set_yticks([])
    for x, c in zip(xs, axes_cols):
        _, info, kind = norms[c]
        if kind == "cat":
            ax.text(x, 1.04, str(info[-1]), ha="center", va="bottom", fontsize=8)
            ax.text(x, -0.04, str(info[0]), ha="center", va="top", fontsize=8)
        else:
            lo, hi = info
            fmt = (lambda v: f"{10**v:.0e}") if kind == "log" else (lambda v: f"{v:.2f}")
            ax.text(x, 1.04, fmt(hi), ha="center", va="bottom", fontsize=8)
            ax.text(x, -0.04, fmt(lo), ha="center", va="top", fontsize=8)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(m_lo, m_hi))
    fig.colorbar(sm, ax=ax, label=metric_col, pad=0.02)
    ax.set_title(f"Parallel coordinates — colour = {metric_col}")
    plt.tight_layout()
    return fig, ax


parallel_coordinates(sweep, ["optimizer", "lr", "weight_decay", "dropout", "batch_size"],
                     "val_loss", log_axes=["lr", "weight_decay"], cmap="viridis_r")
plt.show()


**Reading.** Dark (low-loss) lines bundle through `adamw` at the optimizer axis,
swing through the mid-range of `lr`, and end up in modest dropout. Light (high-loss)
lines come from SGD with extreme `lr`. The lines mostly **cross randomly** between
`weight_decay` and `batch_size`, confirming the slice-plot conclusion that these axes
don't drive the metric here.

## Solution 3 — Specification for the next sweep

A defensible plan, given the evidence:

- **Fix `optimizer = adamw`.** `sgd` and `adam` are dominated in every region of the
  current sweep.
- **Narrow `lr` to `[2e-4, 3e-3]` log-uniform.** This is the bottom of the U-curve.
- **Narrow `dropout` to `[0.1, 0.4] uniform`.** The minimum is inside this band.
- **Fix `weight_decay = 1e-4`.** The chart shows no signal across four orders of
  magnitude; not worth more budget yet.
- **Fix `batch_size = 128`.** Mild effect, larger batches were slightly worse; pick the
  middle value and move on.
- **Budget: 30 trials**, random sampling, on the narrowed space. We expect the
  improvement to come from `lr`/`dropout` interactions that we couldn't see in the
  coarser first sweep.
